# Reviewed executed submission — synthetic prototype
The saved run uses 165 handloom images and MODE="proxy". A standalone MODE="real" assignment below is overwritten by the next configuration cell. Historical source and numeric outputs are preserved; image/rich outputs were removed for sharing.
Real colorway and different-design/similar-palette evaluation remains outstanding. Read README.md and results/RESULTS.md. These results were extracted from saved outputs, not independently reproduced.


In [1]:
import sys
import subprocess
import importlib.util

packages = {
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm"
}

missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

import io
import json
import random
import hashlib
import zipfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps, ImageDraw, ImageEnhance
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import (
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    f1_score,
    confusion_matrix
)

from google.colab import files

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BASE = Path("/content/deeplure")
DATA = BASE / "data"
OUT = BASE / "outputs"

DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
Device: cpu


In [2]:
uploaded = files.upload()

archives = [
    Path(name).resolve()
    for name in uploaded
    if name.lower().endswith(".zip")
]

assert archives, "Please upload the dataset ZIP files."

active_roots = []

for archive in archives:
    destination = DATA / archive.stem.replace(" ", "_")
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive) as z:
        members = [
            member
            for member in z.infolist()
            if not member.is_dir()
        ]

        print(archive.name, ":", len(members), "files")

        if not members:
            print("Empty archive: skipped.")
            continue

        for member in members:
            target = (destination / member.filename).resolve()

            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")

            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(z.read(member))

    active_roots.append(destination)

del uploaded

assert active_roots, "No usable dataset files were extracted."

print("Extraction completed.")

<IPython.core.display.HTML object>

Saving normal_sarees-20261005T085924Z-1-001.zip to normal_sarees-20261005T085924Z-1-001.zip
Saving handloom_sarees-20261005T085913Z-1-001.zip to handloom_sarees-20261005T085913Z-1-001.zip
normal_sarees-20261005T085924Z-1-001.zip : 0 files
Empty archive: skipped.
handloom_sarees-20261005T085913Z-1-001.zip : 165 files
Extraction completed.


In [3]:
image_paths = sorted({
    path
    for root in active_roots
    for path in root.rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
})

rows = []
corrupt = []

for path in tqdm(image_paths):
    try:
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")

            pixel_hash = hashlib.sha256(
                str(image.size).encode() + image.tobytes()
            ).hexdigest()

            small = np.asarray(
                image.convert("L").resize((9, 8))
            )

            bits = (small[:, 1:] > small[:, :-1]).ravel()

            dhash = int(
                "".join("1" if value else "0" for value in bits),
                2
            )

            width, height = image.size

        category = path.parent.name

        if ".rf." in path.name:
            source_key = category + "/" + path.name.split(".rf.")[0]
        else:
            source_key = str(path.relative_to(DATA))

        rows.append({
            "path": str(path),
            "category": category,
            "source_key": source_key,
            "pixel_hash": pixel_hash,
            "dhash": dhash,
            "width": width,
            "height": height
        })

    except Exception as error:
        corrupt.append({
            "path": str(path),
            "error": str(error)
        })

raw = pd.DataFrame(rows)

assert len(raw), "No readable images found."

parent = list(range(len(raw)))

def find_root(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index

def merge_groups(a, b):
    parent[find_root(b)] = find_root(a)

for column in ["source_key", "pixel_hash"]:
    for indices in raw.groupby(column).groups.values():
        indices = list(indices)

        for index in indices[1:]:
            merge_groups(indices[0], index)

raw["source_group"] = [
    f"S{find_root(index):05d}"
    for index in range(len(raw))
]

df = raw.drop_duplicates("source_group").reset_index(drop=True).copy()

df["image_id"] = [
    f"I{index:05d}"
    for index in range(len(df))
]

raw.to_csv(OUT / "all_images.csv", index=False)
df.to_csv(OUT / "inventory.csv", index=False)

hashes = [int(value) for value in df["dhash"]]
image_ids = df["image_id"].tolist()

candidates = []

for i in range(len(df)):
    for j in range(i):
        distance = (hashes[i] ^ hashes[j]).bit_count()

        if distance <= 3:
            candidates.append({
                "image_id_a": image_ids[j],
                "image_id_b": image_ids[i],
                "distance": distance
            })

pd.DataFrame(
    candidates,
    columns=["image_id_a", "image_id_b", "distance"]
).to_csv(OUT / "duplicate_review.csv", index=False)

print("Readable images:", len(raw))
print("Source representatives:", len(df))
print("Corrupt images:", len(corrupt))
print("Possible duplicate pairs:", len(candidates))
print(df.groupby("category").size())

  0%|          | 0/165 [00:00<?, ?it/s]

Readable images: 165
Source representatives: 165
Corrupt images: 0
Possible duplicate pairs: 0
category
handloom_sarees    165
dtype: int64


In [4]:
SHEETS = OUT / "contact_sheets"
SHEETS.mkdir(exist_ok=True)

for start in range(0, len(df), 60):
    page = Image.new("RGB", (1200, 1320), "white")
    draw = ImageDraw.Draw(page)

    subset = df.iloc[start:start + 60]

    for position, row in enumerate(subset.itertuples()):
        with Image.open(row.path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            thumbnail = ImageOps.contain(image, (115, 185))

        x = (position % 10) * 120
        y = (position // 10) * 220

        page.paste(thumbnail, (x, y))
        draw.text((x, y + 188), row.image_id, fill="black")
        draw.text((x, y + 202), row.category[:16], fill="black")

    page.save(SHEETS / f"page_{start // 60 + 1:02d}.jpg")

annotations_path = OUT / "annotations.csv"

if not annotations_path.exists():
    template = df[["image_id", "source_group"]].rename(
        columns={"source_group": "source_id"}
    ).copy()

    template["design_id"] = ""
    template["colorway_id"] = ""
    template["reviewed"] = ""

    template.to_csv(annotations_path, index=False)

labeling_zip = BASE / "private_labeling_files.zip"

with zipfile.ZipFile(
    labeling_zip, "w", zipfile.ZIP_DEFLATED
) as z:
    z.write(annotations_path, "annotations.csv")
    z.write(OUT / "duplicate_review.csv", "duplicate_review.csv")

    for path in SHEETS.glob("*.jpg"):
        z.write(path, f"contact_sheets/{path.name}")

files.download(str(labeling_zip))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
MODE = "real"

In [6]:
MODE = "proxy"
EPOCHS = 5
BATCH_SIZE = 16
EMBED_DIM = 128

if MODE == "real":
    print("Upload the completed annotations.csv")

    uploaded_annotations = files.upload()

    csv_names = [
        name
        for name in uploaded_annotations
        if name.lower().endswith(".csv")
    ]

    assert len(csv_names) == 1, "Upload exactly one annotation CSV."

    annotations_path.write_bytes(
        uploaded_annotations[csv_names[0]]
    )

    annotations = pd.read_csv(
        annotations_path, dtype=str
    ).fillna("")

    assert not annotations["image_id"].duplicated().any()

    annotations = annotations[
        annotations["reviewed"].str.lower().isin(
            ["yes", "true", "1"]
        )
    ].copy()

    assert len(annotations), "No reviewed rows found."

    for column in ["design_id", "source_id"]:
        assert annotations[column].str.strip().ne("").all()

    assert set(annotations["image_id"]).issubset(
        set(df["image_id"])
    )

    assert (
        annotations.groupby("source_id")["design_id"]
        .nunique().max() == 1
    ), "A source photograph has conflicting design labels."

    work = df.merge(
        annotations[
            ["image_id", "source_id", "design_id", "colorway_id"]
        ],
        on="image_id"
    )

    work = work.drop_duplicates("source_id").copy()

    counts = work.groupby("design_id")["source_id"].nunique()
    eligible = counts[counts >= 2].index

    work = work[work["design_id"].isin(eligible)].copy()
    work["identity"] = work["design_id"]

else:
    assert MODE == "proxy"

    work = df.copy()
    work["source_id"] = work["source_group"]
    work["identity"] = work["source_group"]
    work["design_id"] = ""
    work["colorway_id"] = ""

identities = sorted(work["identity"].unique())

assert len(identities) >= 6, (
    "Need at least six eligible identities. "
    "Real mode needs two independent photographs per design."
)

rng = np.random.default_rng(SEED)
rng.shuffle(identities)

holdout_count = max(2, int(len(identities) * 0.15))

if len(identities) - 2 * holdout_count < 2:
    holdout_count = 2

split_map = {}

for index, identity in enumerate(identities):
    if index < holdout_count:
        split_map[identity] = "val"
    elif index < 2 * holdout_count:
        split_map[identity] = "test"
    else:
        split_map[identity] = "train"

work["split"] = work["identity"].map(split_map)

assert work.groupby("source_id")["split"].nunique().max() == 1

RUN = OUT / MODE
RUN.mkdir(exist_ok=True)

work.to_csv(RUN / "split_manifest.csv", index=False)

print("Mode:", MODE)

print(
    work.groupby("split").agg(
        images=("image_id", "size"),
        identities=("identity", "nunique")
    )
)

Mode: proxy
       images  identities
split                    
test       24          24
train     117         117
val        24          24


In [7]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def load_image(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

def square_image(image):
    return ImageOps.pad(
        image,
        (224, 224),
        method=Image.Resampling.BILINEAR,
        color=(127, 127, 127)
    )

eval_transform = T.Compose([
    T.Lambda(square_image),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

train_transform = T.Compose([
    T.Lambda(square_image),
    T.RandomResizedCrop(
        224, scale=(0.8, 1.0), ratio=(0.9, 1.1)
    ),
    T.RandomRotation(8, fill=127),
    T.ColorJitter(
        brightness=0.3,
        contrast=0.4,
        saturation=0.8,
        hue=0.45
    ),
    T.RandomGrayscale(p=0.3),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

def recolor(image, image_id):
    seed = int(
        hashlib.sha256(image_id.encode()).hexdigest()[:8], 16
    )

    local_rng = np.random.default_rng(seed)

    hue = float(
        local_rng.choice([-1, 1]) * local_rng.uniform(0.2, 0.45)
    )

    image = TF.adjust_hue(image, hue)

    image = ImageEnhance.Color(image).enhance(
        float(local_rng.uniform(0.35, 1.7))
    )

    return ImageEnhance.Contrast(image).enhance(
        float(local_rng.uniform(0.8, 1.2))
    )

def make_evaluation_sets(partition):
    subset = work[work["split"] == partition]

    gallery_rows = []
    query_rows = []

    for identity, group in subset.groupby("identity", sort=True):
        group = group.sort_values("image_id")

        if MODE == "proxy":
            row = group.iloc[0].to_dict()

            gallery_rows.append({
                **row,
                "variant": "original",
                "cross_color": False
            })

            query_rows.append({
                **row,
                "variant": "synthetic",
                "cross_color": False
            })

        else:
            known_colors = group[
                group["colorway_id"] != ""
            ]["colorway_id"].value_counts()

            if len(known_colors):
                reference = group[
                    group["colorway_id"] == known_colors.index[-1]
                ].iloc[0]
            else:
                reference = group.iloc[0]

            gallery_rows.append({
                **reference.to_dict(),
                "variant": "original",
                "cross_color": False
            })

            remaining = group[
                group["source_id"] != reference["source_id"]
            ]

            for _, row in remaining.iterrows():
                cross_color = bool(
                    reference["colorway_id"]
                    and row["colorway_id"]
                    and reference["colorway_id"] != row["colorway_id"]
                )

                query_rows.append({
                    **row.to_dict(),
                    "variant": "original",
                    "cross_color": cross_color
                })

    return pd.DataFrame(gallery_rows), pd.DataFrame(query_rows)

val_gallery, val_queries = make_evaluation_sets("val")
test_gallery, test_queries = make_evaluation_sets("test")

for name, table in [
    ("val_gallery", val_gallery),
    ("val_queries", val_queries),
    ("test_gallery", test_gallery),
    ("test_queries", test_queries)
]:
    assert len(table)
    table.to_csv(RUN / f"{name}.csv", index=False)

print("Validation:", len(val_gallery), "gallery,", len(val_queries), "queries")
print("Test:", len(test_gallery), "gallery,", len(test_queries), "queries")
print("Real cross-color test queries:", int(test_queries["cross_color"].sum()))

Validation: 24 gallery, 24 queries
Test: 24 gallery, 24 queries
Real cross-color test queries: 0


In [8]:
class SareeModel(nn.Module):
    def __init__(self, embedding_dim=None, pretrained=True):
        super().__init__()

        weights = (
            ResNet18_Weights.IMAGENET1K_V1
            if pretrained else None
        )

        self.backbone = resnet18(weights=weights)

        if embedding_dim is None:
            self.backbone.fc = nn.Identity()
        else:
            self.backbone.fc = nn.Linear(512, embedding_dim)

    def forward(self, images):
        features = self.backbone(images)
        return F.normalize(features, p=2, dim=1)

class EvaluationDataset(Dataset):
    def __init__(self, table, grayscale=False):
        self.table = table.reset_index(drop=True)
        self.grayscale = grayscale

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        image = load_image(row["path"])

        if row.get("variant", "original") == "synthetic":
            image = recolor(image, row["image_id"])

        if self.grayscale:
            image = ImageOps.grayscale(image).convert("RGB")

        return eval_transform(image)

@torch.inference_mode()
def extract_embeddings(model, table, grayscale=False):
    model.eval()

    loader = DataLoader(
        EvaluationDataset(table, grayscale),
        batch_size=32,
        shuffle=False,
        num_workers=0
    )

    vectors = []

    for images in loader:
        vectors.append(
            model(images.to(DEVICE)).cpu()
        )

    return torch.cat(vectors).numpy()

In [9]:
def retrieval_metrics(scores, relevant):
    order = np.argsort(-scores, axis=1, kind="stable")
    ranked_relevance = np.take_along_axis(relevant, order, axis=1)

    positions = np.arange(1, scores.shape[1] + 1)
    precision = np.cumsum(ranked_relevance, axis=1) / positions

    average_precision = (
        (precision * ranked_relevance).sum(axis=1)
        / np.maximum(ranked_relevance.sum(axis=1), 1)
    )

    return {
        "Recall@1": float(
            ranked_relevance[:, :1].any(axis=1).mean()
        ),
        "Recall@5": float(
            ranked_relevance[:, :min(5, scores.shape[1])]
            .any(axis=1).mean()
        ),
        "mAP": float(average_precision.mean()),
        "queries": int(scores.shape[0]),
        "gallery_size": int(scores.shape[1])
    }

def select_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(
        labels, scores
    )

    f1 = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )

    return float(thresholds[np.argmax(f1)])

def verification_metrics(labels, scores, threshold):
    predictions = scores >= threshold

    tn, fp, fn, tp = confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).ravel()

    return {
        "ROC_AUC": float(roc_auc_score(labels, scores)),
        "F1": float(f1_score(labels, predictions, zero_division=0)),
        "FAR": float(fp / max(fp + tn, 1)),
        "FRR": float(fn / max(fn + tp, 1)),
        "threshold": float(threshold),
        "positive_pairs": int(labels.sum()),
        "negative_pairs": int((~labels).sum()),
        "confusion_matrix": [
            [int(tn), int(fp)],
            [int(fn), int(tp)]
        ]
    }

def evaluate(model, gallery, queries, grayscale=False, threshold=None):
    gallery_vectors = extract_embeddings(model, gallery, grayscale)
    query_vectors = extract_embeddings(model, queries, grayscale)

    scores = query_vectors @ gallery_vectors.T

    relevant = (
        queries["identity"].to_numpy()[:, None]
        == gallery["identity"].to_numpy()[None, :]
    )

    assert relevant.any(axis=1).all()

    labels = relevant.ravel()

    if threshold is None:
        threshold = select_threshold(labels, scores.ravel())

    cross_color = queries["cross_color"].to_numpy(dtype=bool)

    results = {
        "retrieval": retrieval_metrics(scores, relevant),
        "verification": verification_metrics(
            labels, scores.ravel(), threshold
        ),
        "real_cross_color_retrieval": (
            retrieval_metrics(scores[cross_color], relevant[cross_color])
            if cross_color.any() else None
        )
    }

    return results, scores, relevant

In [10]:
baseline = SareeModel().to(DEVICE)

rgb_validation, _, _ = evaluate(
    baseline, val_gallery, val_queries
)

gray_validation, _, _ = evaluate(
    baseline, val_gallery, val_queries, grayscale=True
)

print("RGB baseline:")
print(json.dumps(rgb_validation, indent=2))

print("Grayscale baseline:")
print(json.dumps(gray_validation, indent=2))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 84.0MB/s]


RGB baseline:
{
  "retrieval": {
    "Recall@1": 1.0,
    "Recall@5": 1.0,
    "mAP": 1.0,
    "queries": 24,
    "gallery_size": 24
  },
  "verification": {
    "ROC_AUC": 0.9933574879227053,
    "F1": 0.8181818181818182,
    "FAR": 0.0036231884057971015,
    "FRR": 0.25,
    "threshold": 0.872571587562561,
    "positive_pairs": 24,
    "negative_pairs": 552,
    "confusion_matrix": [
      [
        550,
        2
      ],
      [
        6,
        18
      ]
    ]
  },
  "real_cross_color_retrieval": null
}
Grayscale baseline:
{
  "retrieval": {
    "Recall@1": 1.0,
    "Recall@5": 1.0,
    "mAP": 1.0,
    "queries": 24,
    "gallery_size": 24
  },
  "verification": {
    "ROC_AUC": 0.9990187198067633,
    "F1": 0.9387755102040817,
    "FAR": 0.0036231884057971015,
    "FRR": 0.041666666666666664,
    "threshold": 0.9374255537986755,
    "positive_pairs": 24,
    "negative_pairs": 552,
    "confusion_matrix": [
      [
        550,
        2
      ],
      [
        1,
        23
 

In [11]:
training_data = work[
    work["split"] == "train"
].reset_index(drop=True)

training_identities = sorted(
    training_data["identity"].unique()
)

training_groups = {
    identity: group.reset_index(drop=True)
    for identity, group in training_data.groupby("identity")
}

class TrainingDataset(Dataset):
    def __len__(self):
        return len(training_identities) * 4

    def __getitem__(self, index):
        label = index % len(training_identities)
        identity = training_identities[label]
        group = training_groups[identity]

        anchor = group.iloc[random.randrange(len(group))]

        positives = group[
            group["source_id"] != anchor["source_id"]
        ]

        if len(positives) and anchor["colorway_id"]:
            cross_color = positives[
                (positives["colorway_id"] != "")
                & (
                    positives["colorway_id"]
                    != anchor["colorway_id"]
                )
            ]

            if len(cross_color):
                positives = cross_color

        if len(positives):
            positive = positives.iloc[
                random.randrange(len(positives))
            ]
        else:
            positive = anchor

        image_a = train_transform(load_image(anchor["path"]))
        image_b = train_transform(load_image(positive["path"]))

        return image_a, image_b, label

def batch_triplet_loss(z1, z2, labels, margin=0.2):
    distances = torch.cdist(z1, z2, p=2)

    valid_negatives = labels[:, None] != labels[None, :]
    valid_rows = valid_negatives.any(dim=1)

    if not valid_rows.any():
        return (z1.sum() + z2.sum()) * 0

    negative_distance = distances.masked_fill(
        ~valid_negatives, float("inf")
    ).min(dim=1).values

    positive_distance = (z1 - z2).norm(p=2, dim=1)

    return F.relu(
        positive_distance[valid_rows]
        - negative_distance[valid_rows]
        + margin
    ).mean()

generator = torch.Generator().manual_seed(SEED)

training_loader = DataLoader(
    TrainingDataset(),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=generator
)

In [12]:
model = SareeModel(EMBED_DIM).to(DEVICE)

for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.backbone.layer4.parameters():
    parameter.requires_grad = True

for parameter in model.backbone.fc.parameters():
    parameter.requires_grad = True

optimizer = torch.optim.AdamW([
    {
        "params": model.backbone.layer4.parameters(),
        "lr": 1e-4
    },
    {
        "params": model.backbone.fc.parameters(),
        "lr": 1e-3
    }
], weight_decay=1e-4)

history = []
best_score = (-1.0, -1.0)
checkpoint_path = RUN / "best_model.pt"

for epoch in range(EPOCHS):
    model.train()

    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    total_loss = 0.0
    sample_count = 0

    for images_a, images_b, labels in tqdm(
        training_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    ):
        images_a = images_a.to(DEVICE)
        images_b = images_b.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        vectors = model(
            torch.cat([images_a, images_b], dim=0)
        )

        z1, z2 = vectors.chunk(2)

        loss = (
            batch_triplet_loss(z1, z2, labels)
            + batch_triplet_loss(z2, z1, labels)
        ) / 2

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(images_a)
        sample_count += len(images_a)

    validation, _, _ = evaluate(
        model, val_gallery, val_queries
    )

    score = (
        validation["retrieval"]["Recall@1"],
        validation["retrieval"]["mAP"]
    )

    record = {
        "epoch": epoch + 1,
        "loss": total_loss / max(sample_count, 1),
        **validation["retrieval"]
    }

    history.append(record)
    print(record)

    if score > best_score:
        best_score = score

        torch.save({
            "state_dict": model.state_dict(),
            "embedding_dim": EMBED_DIM,
            "epoch": epoch + 1,
            "mode": MODE
        }, checkpoint_path)

    pd.DataFrame(history).to_csv(
        RUN / "training_history.csv", index=False
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["state_dict"])
model.eval()

print("Best epoch:", checkpoint["epoch"])

Epoch 1/5:   0%|          | 0/30 [00:00<?, ?it/s]

{'epoch': 1, 'loss': 0.05502088643355757, 'Recall@1': 0.9583333333333334, 'Recall@5': 1.0, 'mAP': 0.9791666666666666, 'queries': 24, 'gallery_size': 24}


Epoch 2/5:   0%|          | 0/30 [00:00<?, ?it/s]

{'epoch': 2, 'loss': 0.028969787889056735, 'Recall@1': 1.0, 'Recall@5': 1.0, 'mAP': 1.0, 'queries': 24, 'gallery_size': 24}


Epoch 3/5:   0%|          | 0/30 [00:00<?, ?it/s]

{'epoch': 3, 'loss': 0.027445546868774626, 'Recall@1': 0.9583333333333334, 'Recall@5': 1.0, 'mAP': 0.9791666666666666, 'queries': 24, 'gallery_size': 24}


Epoch 4/5:   0%|          | 0/30 [00:00<?, ?it/s]

{'epoch': 4, 'loss': 0.016765470035437845, 'Recall@1': 0.9583333333333334, 'Recall@5': 1.0, 'mAP': 0.9791666666666666, 'queries': 24, 'gallery_size': 24}


Epoch 5/5:   0%|          | 0/30 [00:00<?, ?it/s]

{'epoch': 5, 'loss': 0.016493222986658413, 'Recall@1': 0.9583333333333334, 'Recall@5': 1.0, 'mAP': 0.9722222222222223, 'queries': 24, 'gallery_size': 24}
Best epoch: 2


In [13]:
experiments = [
    ("pretrained_rgb", baseline, False),
    ("pretrained_gray", baseline, True),
    ("finetuned", model, False)
]

results = {}
summary_rows = []

for name, network, grayscale in experiments:
    validation, _, _ = evaluate(
        network,
        val_gallery,
        val_queries,
        grayscale=grayscale
    )

    threshold = validation["verification"]["threshold"]

    test, scores, relevant = evaluate(
        network,
        test_gallery,
        test_queries,
        grayscale=grayscale,
        threshold=threshold
    )

    results[name] = {
        "validation": validation,
        "test": test
    }

    summary_rows.append({
        "model": name,
        **test["retrieval"],
        "ROC_AUC": test["verification"]["ROC_AUC"],
        "F1": test["verification"]["F1"]
    })

    if name == "finetuned":
        FINAL_THRESHOLD = threshold
        final_scores = scores
        final_relevant = relevant

report = {
    "mode": MODE,
    "claim": (
        "Synthetic source-instance evaluation only."
        if MODE == "proxy"
        else "Reviewed design-disjoint evaluation."
    ),
    "results": results
}

(RUN / "metrics.json").write_text(
    json.dumps(report, indent=2)
)

summary = pd.DataFrame(summary_rows)
summary.to_csv(RUN / "results_summary.csv", index=False)

print(summary.to_string(index=False))

print(
    "Real cross-color retrieval:",
    results["finetuned"]["test"]["real_cross_color_retrieval"]
)

fpr, tpr, _ = roc_curve(
    final_relevant.ravel(),
    final_scores.ravel()
)

plt.plot(fpr, tpr, label="Fine-tuned model")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False acceptance rate")
plt.ylabel("True acceptance rate")
plt.title(f"Verification ROC — {MODE}")
plt.legend()
plt.savefig(RUN / "roc.png", bbox_inches="tight")
plt.show()

predicted_indices = final_scores.argmax(axis=1)
predicted_ids = test_gallery.iloc[
    predicted_indices
]["identity"].to_numpy()

errors = test_queries[
    ["image_id", "identity", "path"]
].copy()

errors["predicted_identity"] = predicted_ids
errors = errors[
    errors["identity"] != errors["predicted_identity"]
]

errors.to_csv(RUN / "retrieval_errors.csv", index=False)

print("Retrieval errors:", len(errors))

          model  Recall@1  Recall@5  mAP  queries  gallery_size  ROC_AUC       F1
 pretrained_rgb       1.0       1.0  1.0       24            24 0.994188 0.790698
pretrained_gray       1.0       1.0  1.0       24            24 0.999396 0.956522
      finetuned       1.0       1.0  1.0       24            24 0.993584 0.690909
Real cross-color retrieval: None


<Figure size 640x480 with 1 Axes>

Retrieval errors: 0


In [14]:
GALLERY = test_gallery.reset_index(drop=True).copy()
GALLERY_VECTORS = extract_embeddings(model, GALLERY)

np.save(RUN / "gallery_embeddings.npy", GALLERY_VECTORS)
GALLERY.to_csv(RUN / "gallery_metadata.csv", index=False)

@torch.inference_mode()
def encode_image(path):
    model.eval()

    image = eval_transform(
        load_image(path)
    ).unsqueeze(0).to(DEVICE)

    return model(image).cpu().numpy()[0]

def search_saree(image_path, top_k=5):
    vector = encode_image(image_path)
    similarities = GALLERY_VECTORS @ vector

    matches = GALLERY[
        ["image_id", "identity", "path"]
    ].copy()

    matches["cosine_similarity"] = similarities

    return (
        matches.sort_values(
            "cosine_similarity", ascending=False
        )
        .drop_duplicates("identity")
        .head(top_k)
        .reset_index(drop=True)
    )

def verify_sarees(path_a, path_b):
    vector_a = encode_image(path_a)
    vector_b = encode_image(path_b)

    similarity = float(vector_a @ vector_b)

    return {
        "cosine_similarity": similarity,
        "threshold": FINAL_THRESHOLD,
        "same_design_prediction": similarity >= FINAL_THRESHOLD,
        "scope": (
            "Prototype threshold from synthetic pairs"
            if MODE == "proxy"
            else "Threshold from reviewed validation design pairs"
        )
    }

print("Identification and verification functions are ready.")

Identification and verification functions are ready.


In [15]:
uploaded_queries = files.upload()

query_paths = [
    Path(name).resolve()
    for name in uploaded_queries
    if Path(name).suffix.lower() in {
        ".jpg", ".jpeg", ".png", ".webp"
    }
]

for path in query_paths:
    print("Query:", path.name)
    print(search_saree(path, top_k=5).to_string(index=False))

if len(query_paths) >= 2:
    print(
        verify_sarees(
            query_paths[0],
            query_paths[1]
        )
    )

<IPython.core.display.HTML object>

Saving image10_jpeg.rf.a8e3a4e500b0ff908c2dd8f682224d67.jpg to image10_jpeg.rf.a8e3a4e500b0ff908c2dd8f682224d67.jpg
Query: image10_jpeg.rf.a8e3a4e500b0ff908c2dd8f682224d67.jpg
image_id identity                                                                                          path  cosine_similarity
  I00119   S00119  /content/deeplure/data/handloom_sarees-20261005T085913Z-1-001/handloom_sarees/img_694883.jpg           0.822485
  I00004   S00004 /content/deeplure/data/handloom_sarees-20261005T085913Z-1-001/handloom_sarees/h_img_35259.jpg           0.796346
  I00123   S00123  /content/deeplure/data/handloom_sarees-20261005T085913Z-1-001/handloom_sarees/img_726717.jpg           0.732054
  I00054   S00054  /content/deeplure/data/handloom_sarees-20261005T085913Z-1-001/handloom_sarees/img_385960.jpg           0.706748
  I00117   S00117   /content/deeplure/data/handloom_sarees-20261005T085913Z-1-001/handloom_sarees/img_68235.jpg           0.688368


In [16]:
def palette_vector(path):
    pixels = np.asarray(
        load_image(path).resize((64, 64))
    ).reshape(-1, 3)

    histogram = np.histogramdd(
        pixels,
        bins=(4, 4, 4),
        range=((0, 256), (0, 256), (0, 256))
    )[0].ravel().astype(float)

    return histogram / histogram.sum()

if MODE == "real":
    gallery_colors = np.stack([
        palette_vector(path)
        for path in test_gallery["path"]
    ])

    query_colors = np.stack([
        palette_vector(path)
        for path in test_queries["path"]
    ])

    distances = (
        (
            query_colors[:, None, :]
            - gallery_colors[None, :, :]
        ) ** 2
    ).sum(axis=2)

    distances[final_relevant] = np.inf

    negative_indices = distances.argmin(axis=1)

    negative_scores = final_scores[
        np.arange(len(test_queries)),
        negative_indices
    ]

    hard_negatives = pd.DataFrame({
        "query_id": test_queries["image_id"].to_numpy(),
        "negative_id": test_gallery.iloc[
            negative_indices
        ]["image_id"].to_numpy(),
        "palette_distance": distances[
            np.arange(len(test_queries)),
            negative_indices
        ],
        "cosine_similarity": negative_scores
    })

    hard_negatives.to_csv(
        RUN / "hard_negative_candidates.csv",
        index=False
    )

    print("Candidate pairs:", len(hard_negatives))
    print(
        "False acceptance rate:",
        float((negative_scores >= FINAL_THRESHOLD).mean())
    )

else:
    print("Skipped: real different-design labels are required.")

Skipped: real different-design labels are required.


In [17]:
model.eval()

dummy_input = torch.zeros(
    1, 3, 224, 224, device=DEVICE
)

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

with torch.inference_mode():
    for _ in range(10):
        model(dummy_input)

    latency = []

    for _ in range(30):
        synchronize()
        start = time.perf_counter()

        model(dummy_input)

        synchronize()

        latency.append(
            (time.perf_counter() - start) * 1000
        )

search_latency = []

for _ in range(30):
    start = time.perf_counter()

    np.argsort(
        -(GALLERY_VECTORS @ GALLERY_VECTORS[0])
    )[:5]

    search_latency.append(
        (time.perf_counter() - start) * 1000
    )

efficiency = {
    "parameters": sum(
        parameter.numel()
        for parameter in model.parameters()
    ),
    "trainable_parameters": sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    "embedding_dimensions": EMBED_DIM,
    "embedding_bytes_float32": EMBED_DIM * 4,
    "forward_median_ms": float(np.median(latency)),
    "forward_p95_ms": float(np.percentile(latency, 95)),
    "search_median_ms": float(np.median(search_latency)),
    "gallery_size": len(GALLERY),
    "device": (
        torch.cuda.get_device_name(0)
        if DEVICE.type == "cuda" else "CPU"
    )
}

config = {
    "mode": MODE,
    "embedding_dim": EMBED_DIM,
    "threshold": FINAL_THRESHOLD,
    "image_size": 224,
    "resize": "aspect-preserving padding",
    "padding_rgb": [127, 127, 127],
    "normalization_mean": MEAN,
    "normalization_std": STD,
    "pretrained_weights": "ResNet18_Weights.IMAGENET1K_V1",
    "seed": SEED
}

(RUN / "efficiency.json").write_text(
    json.dumps(efficiency, indent=2)
)

(RUN / "config.json").write_text(
    json.dumps(config, indent=2)
)

reloaded_model = SareeModel(
    EMBED_DIM, pretrained=False
).to(DEVICE)

saved = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=True
)

reloaded_model.load_state_dict(saved["state_dict"])
reloaded_model.eval()

with torch.inference_mode():
    assert torch.allclose(
        model(dummy_input),
        reloaded_model(dummy_input),
        atol=1e-6
    )

print(json.dumps(efficiency, indent=2))
print("Checkpoint reload passed.")

{
  "parameters": 11242176,
  "trainable_parameters": 8459392,
  "embedding_dimensions": 128,
  "embedding_bytes_float32": 512,
  "forward_median_ms": 96.02840749994357,
  "forward_p95_ms": 304.8718403999601,
  "search_median_ms": 0.006705999908263038,
  "gallery_size": 24,
  "device": "CPU"
}
Checkpoint reload passed.


In [18]:
if MODE == "real":
    approach = (
        "Lean ImageNet ResNet18 maps RGB images to 128-D normalized "
        "embeddings. Resize/pad and ImageNet normalization precede "
        "inference. Identity-balanced triplet training prefers real "
        "cross-color positives, with color jitter, grayscale and mild "
        "geometry changes. Cosine similarity ranks designs; a validation "
        "threshold verifies pairs. Design-disjoint gallery/query splits "
        "measure retrieval, verification and cross-color performance."
    )
else:
    approach = (
        "Lean ImageNet ResNet18 produces 128-D normalized embeddings "
        "after RGB resize/pad and ImageNet normalization. Source-balanced "
        "triplet training uses augmented views, color jitter and grayscale. "
        "Cosine similarity ranks references; a validation threshold "
        "verifies pairs. Grouped source splits test synthetic recoloring "
        "only. Real design and cross-colorway recognition remain "
        "unverified without reviewed labels."
    )

assert len(approach) <= 500

(RUN / "approach.txt").write_text(approach)

environment = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True
)

(RUN / "environment.txt").write_text(environment)

readme = (
    f"DeepLure experiment mode: {MODE}\n"
    "Read metrics.json for actual measured results.\n"
    "Source images and contact sheets are excluded from this package.\n"
    "Proxy scores do not establish real design/colorway recognition.\n"
    "Pretraining: torchvision ResNet18 ImageNet1K V1.\n"
    "Keep vendor data private and delete it when the exercise concludes.\n"
    "Submit the runnable notebook with the results and approach note.\n"
)

(RUN / "README.txt").write_text(readme)

package_path = BASE / f"DeepLure_{MODE}_Results_Private.zip"

excluded = {
    "gallery_embeddings.npy",
    "gallery_metadata.csv"
}

with zipfile.ZipFile(
    package_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for path in RUN.iterdir():
        if path.is_file() and path.name not in excluded:
            z.write(path, path.name)

print("Approach length:", len(approach))
print(approach)

files.download(str(package_path))

Approach length: 406
Lean ImageNet ResNet18 produces 128-D normalized embeddings after RGB resize/pad and ImageNet normalization. Source-balanced triplet training uses augmented views, color jitter and grayscale. Cosine similarity ranks references; a validation threshold verifies pairs. Grouped source splits test synthetic recoloring only. Real design and cross-colorway recognition remain unverified without reviewed labels.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
sample = df.sample(min(8, len(df)), random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax in axes.ravel():
    ax.axis("off")

for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(load_image(row["path"]))
    ax.set_title(
        f'{row["image_id"]}\n{row["category"]}',
        fontsize=10
    )

plt.tight_layout()
plt.show()

<Figure size 1600x800 with 8 Axes>

In [20]:
uploaded = files.upload()

query_files = [
    Path(name).resolve()
    for name in uploaded
    if Path(name).suffix.lower() in {
        ".jpg", ".jpeg", ".png", ".webp"
    }
]

assert query_files, "Upload a saree image."

query_path = query_files[0]
matches = search_saree(query_path, top_k=5)

fig, axes = plt.subplots(
    1,
    len(matches) + 1,
    figsize=(4 * (len(matches) + 1), 5)
)

axes[0].imshow(load_image(query_path))
axes[0].set_title("Uploaded query", fontsize=12)
axes[0].axis("off")

for index, row in matches.iterrows():
    ax = axes[index + 1]
    ax.imshow(load_image(row["path"]))
    ax.set_title(
        f'Rank {index + 1}\n'
        f'Identity: {row["identity"]}\n'
        f'Similarity: {row["cosine_similarity"]:.3f}',
        fontsize=10
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

display(matches[["identity", "cosine_similarity"]])

<IPython.core.display.HTML object>

Saving IMG_E9590_2796f3d7-9007-4e6c-923c-b46a3c378b4b_jpg.rf.3a34ee7034af862f7013c16bfb94ee79.jpg to IMG_E9590_2796f3d7-9007-4e6c-923c-b46a3c378b4b_jpg.rf.3a34ee7034af862f7013c16bfb94ee79.jpg


<Figure size 2400x500 with 6 Axes>

  identity  cosine_similarity
0   S00008           0.677570
1   S00119           0.628740
2   S00117           0.599785
3   S00062           0.597718
4   S00157           0.588742

In [21]:
uploaded = files.upload()

pair_paths = [
    Path(name).resolve()
    for name in uploaded
    if Path(name).suffix.lower() in {
        ".jpg", ".jpeg", ".png", ".webp"
    }
]

assert len(pair_paths) == 2, "Upload exactly two saree images."

result = verify_sarees(pair_paths[0], pair_paths[1])

prediction = (
    "SAME DESIGN"
    if result["same_design_prediction"]
    else "DIFFERENT DESIGNS"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, path, title in zip(
    axes,
    pair_paths,
    ["Image A", "Image B"]
):
    ax.imshow(load_image(path))
    ax.set_title(title)
    ax.axis("off")

fig.suptitle(
    f'{prediction}\n'
    f'Similarity: {result["cosine_similarity"]:.3f} | '
    f'Threshold: {result["threshold"]:.3f}',
    fontsize=14
)

plt.tight_layout()
plt.show()

print("Prediction:", prediction)
print("Similarity:", round(result["cosine_similarity"], 4))
print("Threshold:", round(result["threshold"], 4))
print("Evaluation scope:", result["scope"])

<IPython.core.display.HTML object>

Saving images127_jpg.rf.58275150b0d3a9f0bee9d548ae0888e4.jpg to images127_jpg.rf.58275150b0d3a9f0bee9d548ae0888e4.jpg
Saving images139_jpg.rf.40a5b3b2fd46777a7a4784a747e83e87.jpg to images139_jpg.rf.40a5b3b2fd46777a7a4784a747e83e87.jpg


<Figure size 1000x500 with 2 Axes>

Prediction: DIFFERENT DESIGNS
Similarity: 0.4471
Threshold: 0.8899
Evaluation scope: Prototype threshold from synthetic pairs


In [22]:
number_of_examples = min(4, len(test_queries))
top_k = min(3, len(test_gallery))

for query_index in range(number_of_examples):
    query_row = test_queries.iloc[query_index]
    query_image = load_image(query_row["path"])

    if query_row["variant"] == "synthetic":
        query_image = recolor(
            query_image,
            query_row["image_id"]
        )

    ranked_indices = np.argsort(
        -final_scores[query_index]
    )[:top_k]

    fig, axes = plt.subplots(
        1,
        top_k + 1,
        figsize=(4 * (top_k + 1), 4)
    )

    axes[0].imshow(query_image)
    axes[0].set_title(
        f'Test query\nIdentity: {query_row["identity"]}'
    )
    axes[0].axis("off")

    for rank, gallery_index in enumerate(ranked_indices, start=1):
        gallery_row = test_gallery.iloc[gallery_index]

        correct = (
            gallery_row["identity"] == query_row["identity"]
        )

        status = "Correct" if correct else "Incorrect"

        axes[rank].imshow(load_image(gallery_row["path"]))
        axes[rank].set_title(
            f'Rank {rank}: {status}\n'
            f'Score: {final_scores[query_index, gallery_index]:.3f}',
            color="green" if correct else "red"
        )
        axes[rank].axis("off")

    fig.suptitle(f"Evaluation mode: {MODE}")
    plt.tight_layout()
    plt.show()

<Figure size 1600x400 with 4 Axes>

<Figure size 1600x400 with 4 Axes>

<Figure size 1600x400 with 4 Axes>

<Figure size 1600x400 with 4 Axes>

In [23]:
print("MODE:", MODE)

print("\nFinal test results:")
print(json.dumps(
    results["finetuned"]["test"],
    indent=2
))

assert np.isfinite(GALLERY_VECTORS).all()

assert np.allclose(
    np.linalg.norm(GALLERY_VECTORS, axis=1),
    1.0,
    atol=1e-4
)

assert work.groupby("identity")["split"].nunique().max() == 1

if MODE == "real":
    assert not (
        set(test_gallery["source_id"])
        & set(test_queries["source_id"])
    )

print("\nBasic consistency checks passed.")

MODE: proxy

Final test results:
{
  "retrieval": {
    "Recall@1": 1.0,
    "Recall@5": 1.0,
    "mAP": 1.0,
    "queries": 24,
    "gallery_size": 24
  },
  "verification": {
    "ROC_AUC": 0.9935839371980676,
    "F1": 0.6909090909090909,
    "FAR": 0.021739130434782608,
    "FRR": 0.20833333333333334,
    "threshold": 0.8899458050727844,
    "positive_pairs": 24,
    "negative_pairs": 552,
    "confusion_matrix": [
      [
        540,
        12
      ],
      [
        5,
        19
      ]
    ]
  },
  "real_cross_color_retrieval": null
}

Basic consistency checks passed.
